# AnchorDepth — Local Inference Demo

**Bachelor thesis** — Politehnica University of Timișoara, 2026

Simple depth estimation demo on any image using your trained AnchorDepth model.  
Produces a side-by-side comparison: **Input RGB | AnchorDepth (ours) | Depth Pro zero-shot**

**Usage:** Put your image path in Cell 3 and run all cells.

## 1. Setup

In [ ]:
import sys, os, time
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import ToTensor, Normalize
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Add project paths
ROOT = Path(os.getcwd())  # should be d:\AnchorDepth
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "training"))

import depth_pro
from train_nyu_lora import LoRALinear, apply_lora_to_encoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load Models

Loads both **AnchorDepth (your trained model)** and **Depth Pro zero-shot (baseline)**.  
Change `CHECKPOINT` below to point to your best checkpoint.

In [ ]:
# ============================================================
# CONFIGURATION — change these paths as needed
# ============================================================
CHECKPOINT = "checkpoints/selfsup_v20/selfsup_best.pt"  # your trained model
LORA_RANK = 8
LORA_ALPHA = 8.0

# ============================================================
# Load AnchorDepth (trained)
# ============================================================
print("Loading AnchorDepth...")
model_anchor, _ = depth_pro.create_model_and_transforms(device=device)
apply_lora_to_encoder(model_anchor, rank=LORA_RANK, alpha=LORA_ALPHA)

ckpt = torch.load(CHECKPOINT, map_location=device)
state = ckpt["depth_model"] if "depth_model" in ckpt else ckpt
model_anchor.load_state_dict(state, strict=False)
model_anchor.eval()
print(f"✓ AnchorDepth loaded from: {CHECKPOINT}")

# ============================================================
# Load Depth Pro zero-shot (baseline for comparison)
# ============================================================
print("Loading Depth Pro zero-shot...")
model_zs, _ = depth_pro.create_model_and_transforms(device=device)
model_zs.eval()
print("✓ Depth Pro zero-shot loaded")

## 3. Choose your image

Set the path to any RGB image below (JPG or PNG).

In [ ]:
import tkinter as tk
from tkinter import filedialog

# Open file dialog to pick an image
root = tk.Tk()
root.withdraw()  # hide the main window
root.attributes("-topmost", True)  # bring dialog to front
IMAGE_PATH = filedialog.askopenfilename(
    title="Select an image",
    filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp *.tiff"), ("All files", "*.*")]
)
root.destroy()

if not IMAGE_PATH:
    raise RuntimeError("No image selected!")

# Load and display
img = Image.open(IMAGE_PATH).convert("RGB")
print(f"Image: {IMAGE_PATH} ({img.size[0]}x{img.size[1]})")
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title("Input Image", fontsize=14)
plt.axis("off")
plt.show()

## 4. Run Inference + Comparison

In [ ]:
@torch.no_grad()
def predict_depth(model, img_pil, device):
    """Run Depth Pro model on a PIL image -> (depth_map, time_ms)."""
    # Preprocess: resize to 1536x1536, normalize
    img_1536 = img_pil.resize((1536, 1536), Image.LANCZOS)
    norm = Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    inp = norm(ToTensor()(img_1536)).unsqueeze(0).to(device)

    # Warm CUDA
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    with torch.amp.autocast(device.type, dtype=torch.float16):
        canonical_inv_depth, fov_deg = model(inp)
        # Convert canonical inverse depth to metric depth
        f_px = 0.5 * 1536 / torch.tan(0.5 * torch.deg2rad(fov_deg.float()))
        inv_depth = canonical_inv_depth * (1536 / f_px)
        depth = 1.0 / torch.clamp(inv_depth, min=1e-4, max=1e4)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (time.time() - t0) * 1000

    # Resize back to original dimensions
    orig_w, orig_h = img_pil.size
    depth = F.interpolate(depth, size=(orig_h, orig_w), mode="bilinear", align_corners=False)
    depth_np = depth.squeeze().cpu().float().numpy()

    return depth_np, elapsed_ms


def depth_to_colormap(depth, cmap="magma", clip_range=(1.0, 80.0)):
    """Convert depth map to colored image using inverse-depth for better viz."""
    d = np.clip(depth, clip_range[0], clip_range[1])
    inv = 1.0 / d
    lo, hi = np.percentile(inv, [2, 98])
    inv_norm = np.clip((inv - lo) / (hi - lo + 1e-8), 0, 1)
    colormap = cm.get_cmap(cmap)
    rgb = (colormap(inv_norm)[:, :, :3] * 255).astype(np.uint8)
    return rgb


# Run both models
print("Running AnchorDepth...")
depth_anchor, t_anchor = predict_depth(model_anchor, img, device)
print(f"  → {t_anchor:.0f} ms")

print("Running Depth Pro zero-shot...")
depth_zs, t_zs = predict_depth(model_zs, img, device)
print(f"  → {t_zs:.0f} ms")

# Colorize
color_anchor = depth_to_colormap(depth_anchor)
color_zs = depth_to_colormap(depth_zs)

# Plot side-by-side
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(img)
axes[0].set_title("Input Image", fontsize=15, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(color_anchor)
axes[1].set_title(f"AnchorDepth (ours)\n{t_anchor:.0f} ms", fontsize=15, fontweight="bold", color="green")
axes[1].axis("off")

axes[2].imshow(color_zs)
axes[2].set_title(f"Depth Pro (zero-shot)\n{t_zs:.0f} ms", fontsize=15, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("figures/inference_comparison.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("\n✓ Saved to figures/inference_comparison.png")

## 5. (Optional) Save depth maps as separate images

In [ ]:
# Save individual outputs for video/slides
Image.fromarray(color_anchor).save("figures/depth_anchordepth.png")
Image.fromarray(color_zs).save("figures/depth_zeroshot.png")

# Also save raw depth as .npy for analysis
np.save("figures/depth_anchordepth_raw.npy", depth_anchor)
np.save("figures/depth_zeroshot_raw.npy", depth_zs)

print("✓ Saved individual depth maps to figures/")
print(f"  - figures/depth_anchordepth.png")
print(f"  - figures/depth_zeroshot.png")
print(f"  - figures/depth_anchordepth_raw.npy")
print(f"  - figures/depth_zeroshot_raw.npy")